# ReadmitIQ: Clinical Transformer Fine-Tuning (Colab GPU Accelerated)

This notebook fine-tunes a **DistilBERT** (or **ClinicalBERT**) model on synthetic clinical discharge notes to predict 30-day readmission risk on a free GPU (T4/V100/A100) in Google Colab.

### Pipeline Overview:
1. **GPU Verification**: Confirm CUDA hardware acceleration.
2. **Environment Setup**: Install Hugging Face `transformers`, `accelerate`, and `evaluate`.
3. **Data Loading**: Load patient-level split parquets with synthetic discharge notes.
4. **Class Imbalance Handling**: Weighted Cross-Entropy Loss (7.7x weight on positive readmission class).
5. **Mixed Precision Training**: FP16 accelerated fine-tuning.
6. **Evaluation**: Compute AUC-ROC, 95% Bootstrap Confidence Interval, F1, and Precision@top-10% (Lift).

In [ ]:
# Step 1: Check GPU Availability
!nvidia-smi

In [ ]:
# Step 2: Install Dependencies
!pip install -q transformers datasets accelerate scikit-learn pandas pyarrow

In [ ]:
import torch
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AdamW, get_linear_schedule_with_warmup
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score, precision_recall_curve

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Active Training Device: {device}")

In [ ]:
# Step 3: Load Data Splits (Upload train_rag.parquet and val_rag.parquet to Colab or Google Drive)
# If using Google Drive:
# from google.colab import drive
# drive.mount('/content/drive')

train_df = pd.read_parquet("train_rag.parquet")
val_df = pd.read_parquet("val_rag.parquet")

print(f"Loaded Train Encounters: {len(train_df)}, Val Encounters: {len(val_df)}")
print(f"Target Positive Prevalence: {train_df['readmitted_30d'].mean():.4%}")

In [ ]:
class ClinicalNotesDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = list(texts)
        self.labels = list(labels)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            str(self.texts[idx]),
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )
        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "label": torch.tensor(int(self.labels[idx]), dtype=torch.long)
        }

MODEL_NAME = "distilbert-base-uncased"  # Or 'emilyalsentzer/Bio_ClinicalBERT'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_dataset = ClinicalNotesDataset(train_df["clinical_note"], train_df["readmitted_30d"], tokenizer)
val_dataset = ClinicalNotesDataset(val_df["clinical_note"], val_df["readmitted_30d"], tokenizer)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

In [ ]:
# Step 4: Model Initialization with Class-Weighted Loss
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(device)

# 7.7x weight on positive class (approx 11% prevalence)
class_weights = torch.tensor([1.0, 7.7], dtype=torch.float32).to(device)
criterion = torch.nn.CrossEntropyLoss(weight=class_weights)

epochs = 3
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
total_steps = len(train_loader) * epochs
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(total_steps*0.1), num_training_steps=total_steps)

# FP16 Mixed Precision Scaler for GPU acceleration
scaler = torch.cuda.amp.GradScaler(enabled=(device.type == 'cuda'))

In [ ]:
# Step 5: Training Loop with FP16 Acceleration
for epoch in range(epochs):
    model.train()
    total_train_loss = 0.0
    for step, batch in enumerate(train_loader):
        optimizer.zero_grad()
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = criterion(outputs.logits, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        total_train_loss += loss.item()
        if (step + 1) % 100 == 0:
            print(f"Epoch {epoch+1}/{epochs} | Step {step+1}/{len(train_loader)} | Loss: {loss.item():.4f}")

    print(f"--> Completed Epoch {epoch+1} | Average Loss: {total_train_loss / len(train_loader):.4f}")

In [ ]:
# Step 6: Evaluation on Validation Cohort
model.eval()
val_probs = []
val_labels = []

with torch.no_grad():
    for batch in val_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        probs = torch.softmax(outputs.logits, dim=-1)[:, 1]
        val_probs.extend(probs.cpu().numpy())
        val_labels.extend(batch["label"].numpy())

val_probs = np.array(val_probs)
val_labels = np.array(val_labels)

auc = roc_auc_score(val_labels, val_probs)

# 95% Bootstrap Confidence Interval
rng = np.random.RandomState(42)
boot_aucs = [roc_auc_score(val_labels[idx], val_probs[idx]) for idx in [rng.randint(0, len(val_labels), len(val_labels)) for _ in range(1000)]]
ci_lower, ci_upper = np.percentile(boot_aucs, [2.5, 97.5])

# Precision@top-10% and Lift
k_10 = int(len(val_labels) * 0.10)
top_10_idx = np.argsort(val_probs)[::-1][:k_10]
prec_top_10 = np.mean(val_labels[top_10_idx])
lift = prec_top_10 / np.mean(val_labels)

print("="*50)
print(f"DistilBERT Validation AUC-ROC: {auc:.4f} [95% CI: {ci_lower:.4f} - {ci_upper:.4f}]")
print(f"Precision@top-10%: {prec_top_10:.4f}")
print(f"Lift@top-10%: {lift:.2f}x")
print("="*50)

In [ ]:
# Step 7: Save Model Weights Artifact
model.save_pretrained("distilbert_readmit_model")
tokenizer.save_pretrained("distilbert_readmit_model")
!zip -r distilbert_readmit_model.zip distilbert_readmit_model
print("Model artifact zipped and ready to download!")